In [3]:
import ROOT as r
import math
import numpy as np
from itertools import combinations, permutations
from neutrinosolver import *
from collections import Counter

# opens file and tree
f = r.TFile("/home/masha/actual_data/RunIISummer20UL17NanoAODv9-2_TTtoLNu2Q-1Jets-smeft_MTT-0to700_TuneCP5_13TeV_madgraphMLM-pythia8-2_NANOAODSIM106X_mc2017.root")
tree = f.Get("Events")

nEvents = tree.GetEntries()

print("Events in file:", nEvents)

# classes for muon, electron, jets

class MyMuon(r.TLorentzVector):
    def __init__(self, pt=0, eta=0, phi=0, mass=0, iso=0.0, charge=0):
        super().__init__()
        self.SetPtEtaPhiM(pt, eta, phi, mass)
        self.isolation = iso
        self.charge = charge
    
    
    def IsIsolated(self, relcut=0.1):
        if self.Pt() == 0:
            return False
        return self.isolation < relcut

class MyElectron(r.TLorentzVector):
    def __init__(self, pt=0, eta=0, phi=0, mass=0, iso=0.0, charge=0, cutBased=0):
        super().__init__()
        self.SetPtEtaPhiM(pt, eta, phi, mass)
        self.isolation = iso
        self.charge = charge
        self.cutBased = cutBased

    def IsIsolated(self, relcut=0.1):
        if self.Pt() == 0:
            return False
        return self.isolation < relcut

class MyJet(r.TLorentzVector):
    def __init__(self, pt=0, eta=0, phi=0, mass=0, btag=0.0, jetid=False):
        super().__init__()
        self.SetPtEtaPhiM(pt, eta, phi, mass)
        self.btag = btag
        self.jetid = jetid
        self.is_btagged = False

    def IsBTagged(self, threshold):
        return self.btag > threshold

    def HasJetID(self):
        return (self.jetid & 2) != 0
    
    

# histograms

h_Mwh_vs_Mth = r.TH2F(
    "h_Mwh_vs_Mth",
    "M(W_h) vs M(t_h); M(t_h) [GeV]; M(W_h) [GeV]",
    100, 0, 500,   # 5 GeV x bins
    60, 0, 300     # 5 GeV y bins
)

# Dn,min distributions
h_Dn_correct = r.TH1F(
    "h_Dn_correct",
    "Correct leptonic b;D_{n,min};Normalized events",
    50, 0, 150
)

h_Dn_wrong = r.TH1F(
    "h_Dn_wrong",
    "Wrong leptonic b;D_{n,min};Normalized events",
    50, 0, 150
)


# bh + light jet mass
h_Mbhj_correct = r.TH1F(
    "h_Mbhj_correct",
    "Correct bh+jet;M(b_{h}+j) [GeV];Normalized events",
    50, 0, 500
)

h_Mbhj_wrong = r.TH1F(
    "h_Mbhj_wrong",
    "Wrong bh+jet;M(b_{h}+j) [GeV];Normalized events",
    50, 0, 500
)


# weighted error
h_Mwh_vs_Mth.Sumw2()

h_Dn_correct.Sumw2()
h_Dn_wrong.Sumw2()

h_Mbhj_correct.Sumw2()
h_Mbhj_wrong.Sumw2()


# analysis cuts

cuts = {
    "Muon": {
        "pt_min": 30.0,
        "eta_max": 2.4,
        "iso_max": 0.15
    },
    "Electron": {
        "pt_min": 30.0,
        "eta_max": 2.4,
        "cutBased": 4,
        "iso_max": 0.15
        
    },
    "Jets": {
        "pt_min": 30.0,
        "eta_max": 2.4,
        "btag": 0.3040
    },
    "MET": {
        "pt_min": 20.0
    }
}

cuts["Trigger"] = {
    "muon": ["HLT_IsoMu27"],
    "electron": ["HLT_Ele27_WPTight_Gsf", "HLT_Ele32_WPTight_Gsf"]
}

cutflow = {
    "total": 0,

    # triggers
    "pass_mu_trigger": 0,
    "pass_ele_trigger": 0,

    # lepton selection
    "exactly_1_lepton": 0,

    # MET
    "pass_MET": 0,

    # jets
    "3jets": 0,
    "4plus_jets": 0,

    # b-tag categories
    "4pj_2b": 0,
    "4pj_1b": 0,
    "3j_2b": 0,
}

def make_gen_vector(idx):
    vec = r.TLorentzVector()
    vec.SetPtEtaPhiM(
        tree.GenPart_pt[idx],
        tree.GenPart_eta[idx],
        tree.GenPart_phi[idx],
        tree.GenPart_mass[idx]
    )
    return vec



def make_neutrino(nusol):
    nu_vec = nusol.nu

    nu = r.TLorentzVector()
    nu.SetPxPyPzE(
        nu_vec[0],
        nu_vec[1],
        nu_vec[2],
        math.sqrt(
            nu_vec[0]**2 +
            nu_vec[1]**2 +
            nu_vec[2]**2
        )
    )

    return nu

LAST_COPY_BIT = 13


def get_children(parent_idx, mothers):
    """Return indices whose direct mother is parent_idx."""
    return [
        idx for idx, mother_idx in enumerate(mothers)
        if mother_idx == parent_idx
    ]


def get_mother_pdg(idx, mothers, pdg_ids):
    """
    GenPart_genPartIdxMother contains an INDEX, not a PDG ID.
    This function converts that mother index into its PDG ID.
    """
    mother_idx = mothers[idx]

    if mother_idx < 0 or mother_idx >= len(pdg_ids):
        return None

    return pdg_ids[mother_idx]


def is_last_copy(idx):
    """Check the NanoAOD isLastCopy status flag."""
    flags = int(tree.GenPart_statusFlags[idx])
    return bool(flags & (1 << LAST_COPY_BIT))


def get_copy_chain(idx, mothers, pdg_ids):
    """
    Follow a particle through children with exactly the same PDG ID.
    """
    chain = [idx]
    visited = {idx}

    while True:
        current = chain[-1]

        same_pdg_children = [
            child_idx
            for child_idx in get_children(current, mothers)
            if pdg_ids[child_idx] == pdg_ids[current]
            and child_idx not in visited
        ]

        if len(same_pdg_children) == 0:
            break

        if len(same_pdg_children) > 1:
            print(
                "WARNING: multiple same-PDG children:",
                "parent =", current,
                "pdg =", pdg_ids[current],
                "children =", same_pdg_children
            )

        current = same_pdg_children[0]
        chain.append(current)
        visited.add(current)

    return chain


def get_last_copy(idx, mothers, pdg_ids):
    return get_copy_chain(idx, mothers, pdg_ids)[-1]


def get_decay_products(idx, mothers, pdg_ids):
    chain = get_copy_chain(idx, mothers, pdg_ids)
    chain_set = set(chain)

    products = []

    for parent_idx in chain:
        for child_idx in get_children(parent_idx, mothers):
            if child_idx not in chain_set:
                products.append(child_idx)

    # Remove duplicates while preserving order
    return list(dict.fromkeys(products))


def classify_w_decay(W_idx, mothers, pdg_ids):
    #check if w decay is lep or had
    W_idx = get_last_copy(W_idx, mothers, pdg_ids)
    W_pdg = pdg_ids[W_idx]

    if W_pdg not in (24, -24):
        return None, []

    daughters = get_decay_products(W_idx, mothers, pdg_ids)

    if W_pdg == 24:
        allowed_leptons = {-11, -13, -15}
        allowed_neutrinos = {12, 14, 16}

        allowed_up_quarks = {2, 4}
        allowed_down_quarks = {-1, -3, -5}

    else:
        allowed_leptons = {11, 13, 15}
        allowed_neutrinos = {-12, -14, -16}

        allowed_up_quarks = {-2, -4}
        allowed_down_quarks = {1, 3, 5}

    leptons = [
        idx for idx in daughters
        if pdg_ids[idx] in allowed_leptons
    ]

    neutrinos = [
        idx for idx in daughters
        if pdg_ids[idx] in allowed_neutrinos
    ]

    # Check that the lepton and neutrino have the same flavor
    for lepton_idx in leptons:
        for neutrino_idx in neutrinos:
            if abs(pdg_ids[neutrino_idx]) == abs(pdg_ids[lepton_idx]) + 1:
                return "leptonic", [lepton_idx, neutrino_idx]

    quarks = [
        idx for idx in daughters
        if 1 <= abs(pdg_ids[idx]) <= 5
    ]

    # Check the allowed W quark charges/signs
    for q1_idx, q2_idx in combinations(quarks, 2):
        q1_pdg = pdg_ids[q1_idx]
        q2_pdg = pdg_ids[q2_idx]

        valid_pair = (
            q1_pdg in allowed_up_quarks
            and q2_pdg in allowed_down_quarks
        ) or (
            q2_pdg in allowed_up_quarks
            and q1_pdg in allowed_down_quarks
        )

        if valid_pair:
            q1_last = get_last_copy(q1_idx, mothers, pdg_ids)
            q2_last = get_last_copy(q2_idx, mothers, pdg_ids)

            return "hadronic", [q1_last, q2_last]

    return None, []


def print_gen_table(event, mothers, pdg_ids):
    #to crosscheck
    print(f"\n========== GENERATOR EVENT {event} ==========")

    for idx, pdg_id in enumerate(pdg_ids):
        mother_idx = mothers[idx]
        mother_pdg = get_mother_pdg(idx, mothers, pdg_ids)

        print(
            f"idx={idx:3d}  "
            f"pdg={pdg_id:5d}  "
            f"mother_idx={mother_idx:3d}  "
            f"mother_pdg={str(mother_pdg):>5s}  "
            f"lastCopy={is_last_copy(idx)}"
        )

def has_ancestor(particle_idx, ancestor_idx, mothers):
    """
    Check whether ancestor_idx appears in the generator mother chain
    of particle_idx.
    """
    current = particle_idx
    visited = set()

    while 0 <= current < len(mothers) and current not in visited:

        if current == ancestor_idx:
            return True

        visited.add(current)
        current = mothers[current]

    return False


def unique_jet_parton_match(reco_jets, gen_partons, max_dr=0.4):
    """
    Find the one-to-one reco-jet/gen-parton assignment having the
    smallest total DeltaR.

    The returned jets have the same order as gen_partons.
    """
    if len(reco_jets) < len(gen_partons):
        return None

    best_match = None
    best_total_dr = float("inf")

    for jet_indices in permutations(
        range(len(reco_jets)),
        len(gen_partons)
    ):
        candidate_jets = [
            reco_jets[idx]
            for idx in jet_indices
        ]

        delta_rs = [
            reco_jet.DeltaR(gen_parton)
            for reco_jet, gen_parton
            in zip(candidate_jets, gen_partons)
        ]

        if not all(dr < max_dr for dr in delta_rs):
            continue

        total_dr = sum(delta_rs)

        if total_dr < best_total_dr:
            best_total_dr = total_dr

            best_match = {
                "jets": candidate_jets,
                "delta_rs": delta_rs,
                "total_dr": total_dr
            }

    return best_match

n_3jet_before_solver = 0
n_3jet_solver_success = 0
n_3jet_solver_fail = 0

truth_fail = Counter()
reco_fail = Counter()

# Print the full generator table for only the first two events.
# to crosscheck
DEBUG_GEN_EVENTS = 2

# event loop
for event in range(nEvents):

    cutflow["total"] += 1

    tree.GetEntry(event)
    weight = tree.Generator_weight

    gen_b_had = None
    gen_b_lep = None
    gen_w_quarks = []

    gen_b_had_idx = None
    gen_b_lep_idx = None
    gen_w_quark_idxs = []

    gen_W_lep_idx = None
    gen_W_had_idx = None
    gen_lepton_idx = None
    gen_neutrino_idx = None

    # These are integer indices and integer PDG IDs.
    gen_mothers = [
        int(value)
        for value in tree.GenPart_genPartIdxMother
    ]

    gen_pdg_ids = [
        int(value)
        for value in tree.GenPart_pdgId
    ]

    # Print index, PDG ID, mother index and mother PDG ID.
    if event < DEBUG_GEN_EVENTS:
        print_gen_table(event, gen_mothers, gen_pdg_ids)

    # Explicitly find the last-copy top and antitop.
    top_candidates = [
        idx for idx in range(tree.nGenPart)
        if gen_pdg_ids[idx] == 6
        and is_last_copy(idx)
    ]

    antitop_candidates = [
        idx for idx in range(tree.nGenPart)
        if gen_pdg_ids[idx] == -6
        and is_last_copy(idx)
    ]

    if len(top_candidates) != 1:
        truth_fail["not_exactly_one_last_copy_top"] += 1
        continue

    if len(antitop_candidates) != 1:
        truth_fail["not_exactly_one_last_copy_antitop"] += 1
        continue

    top_idx = top_candidates[0]
    antitop_idx = antitop_candidates[0]

    # Final safety check that these really are t and anti-t.
    if gen_pdg_ids[top_idx] != 6:
        truth_fail["selected_top_has_wrong_pdg"] += 1
        continue

    if gen_pdg_ids[antitop_idx] != -6:
        truth_fail["selected_antitop_has_wrong_pdg"] += 1
        continue

    truth_event_valid = True

    for current_top_idx in (top_idx, antitop_idx):

        current_top_pdg = gen_pdg_ids[current_top_idx]

        # Explicit charge-consistent top decay:
        #
        # t     -> W+ b
        # anti-t -> W- anti-b
        if current_top_pdg == 6:
            expected_W_pdg = 24
            expected_b_pdg = 5

        elif current_top_pdg == -6:
            expected_W_pdg = -24
            expected_b_pdg = -5

        else:
            truth_fail["particle_in_top_loop_is_not_top"] += 1
            truth_event_valid = False
            break

        top_decay_products = get_decay_products(
            current_top_idx,
            gen_mothers,
            gen_pdg_ids
        )

        b_candidates = [
            idx for idx in top_decay_products
            if gen_pdg_ids[idx] == expected_b_pdg
        ]

        W_candidates = [
            idx for idx in top_decay_products
            if gen_pdg_ids[idx] == expected_W_pdg
        ]

        # Require exactly the expected b and W.
        if len(b_candidates) != 1:
            truth_fail["top_does_not_have_exact_expected_b"] += 1
            truth_event_valid = False

            if event < DEBUG_GEN_EVENTS:
                print(
                    "BAD TOP DECAY: expected b PDG",
                    expected_b_pdg,
                    "from top index",
                    current_top_idx,
                    "top PDG",
                    current_top_pdg,
                    "products:",
                    [
                        (idx, gen_pdg_ids[idx])
                        for idx in top_decay_products
                    ]
                )

            break

        if len(W_candidates) != 1:
            truth_fail["top_does_not_have_exact_expected_W"] += 1
            truth_event_valid = False

            if event < DEBUG_GEN_EVENTS:
                print(
                    "BAD TOP DECAY: expected W PDG",
                    expected_W_pdg,
                    "from top index",
                    current_top_idx,
                    "top PDG",
                    current_top_pdg,
                    "products:",
                    [
                        (idx, gen_pdg_ids[idx])
                        for idx in top_decay_products
                    ]
                )

            break

        b_idx = get_last_copy(
            b_candidates[0],
            gen_mothers,
            gen_pdg_ids
        )

        W_idx = get_last_copy(
            W_candidates[0],
            gen_mothers,
            gen_pdg_ids
        )

        # Check again after following the particle copies.
        if gen_pdg_ids[b_idx] != expected_b_pdg:
            truth_fail["final_b_copy_has_wrong_pdg"] += 1
            truth_event_valid = False
            break

        if gen_pdg_ids[W_idx] != expected_W_pdg:
            truth_fail["final_W_copy_has_wrong_pdg"] += 1
            truth_event_valid = False
            break

        W_decay_type, W_decay_indices = classify_w_decay(
            W_idx,
            gen_mothers,
            gen_pdg_ids
        )

        if W_decay_type is None:
            truth_fail["W_decay_failed_pdg_checks"] += 1
            truth_event_valid = False

            if event < DEBUG_GEN_EVENTS:
                W_products = get_decay_products(
                    W_idx,
                    gen_mothers,
                    gen_pdg_ids
                )

                print(
                    "BAD W DECAY:",
                    "W index =", W_idx,
                    "W PDG =", gen_pdg_ids[W_idx],
                    "products =",
                    [
                        (idx, gen_pdg_ids[idx])
                        for idx in W_products
                    ]
                )

            break

        if event < DEBUG_GEN_EVENTS:
            print(
                "\nAccepted top decay:",
                f"top index={current_top_idx}",
                f"top PDG={current_top_pdg}",
                f"b index={b_idx}",
                f"b PDG={gen_pdg_ids[b_idx]}",
                f"W index={W_idx}",
                f"W PDG={gen_pdg_ids[W_idx]}",
                f"W decay={W_decay_type}",
                "W products=",
                [
                    (idx, gen_pdg_ids[idx])
                    for idx in W_decay_indices
                ]
            )

        if W_decay_type == "leptonic":

            if gen_b_lep_idx is not None:
                truth_fail["two_leptonic_top_decays"] += 1
                truth_event_valid = False
                break

            if len(W_decay_indices) != 2:
                truth_fail["bad_leptonic_W_products"] += 1
                truth_event_valid = False
                break

            gen_b_lep_idx = b_idx
            gen_W_lep_idx = W_idx
            gen_lepton_idx = W_decay_indices[0]
            gen_neutrino_idx = W_decay_indices[1]

        elif W_decay_type == "hadronic":

            if gen_b_had_idx is not None:
                truth_fail["two_hadronic_top_decays"] += 1
                truth_event_valid = False
                break

            if len(W_decay_indices) != 2:
                truth_fail["hadronic_W_does_not_have_two_quarks"] += 1
                truth_event_valid = False
                break

            gen_b_had_idx = b_idx
            gen_W_had_idx = W_idx
            gen_w_quark_idxs = W_decay_indices

    if not truth_event_valid:
        continue

    # Require exactly one leptonic and one hadronic top decay.
    if gen_b_lep_idx is None:
        truth_fail["missing_leptonic_b"] += 1
        continue

    if gen_b_had_idx is None:
        truth_fail["missing_hadronic_b"] += 1
        continue

    if len(gen_w_quark_idxs) != 2:
        truth_fail["missing_two_hadronic_W_quarks"] += 1
        continue

    # Final explicit PDG checks.
    if abs(gen_pdg_ids[gen_b_lep_idx]) != 5:
        truth_fail["b_lep_is_not_b_quark"] += 1
        continue

    if abs(gen_pdg_ids[gen_b_had_idx]) != 5:
        truth_fail["b_had_is_not_b_quark"] += 1
        continue

    if not all(
        1 <= abs(gen_pdg_ids[idx]) <= 5
        for idx in gen_w_quark_idxs
    ):
        truth_fail["W_products_are_not_quarks"] += 1
        continue

    gen_b_lep = make_gen_vector(gen_b_lep_idx)
    gen_b_had = make_gen_vector(gen_b_had_idx)

    gen_w_quarks = [
        make_gen_vector(idx)
        for idx in gen_w_quark_idxs
    ]

    if event < DEBUG_GEN_EVENTS:
        print(
            "\nFINAL TRUTH ASSIGNMENT:",
            f"b_lep index={gen_b_lep_idx}",
            f"PDG={gen_pdg_ids[gen_b_lep_idx]}",
            f"b_had index={gen_b_had_idx}",
            f"PDG={gen_pdg_ids[gen_b_had_idx]}",
            "W quarks=",
            [
                (idx, gen_pdg_ids[idx])
                for idx in gen_w_quark_idxs
            ]
        )

    # MET filters
    if not all([
        tree.Flag_goodVertices,
        tree.Flag_globalSuperTightHalo2016Filter,
        tree.Flag_HBHENoiseFilter,
        tree.Flag_HBHENoiseIsoFilter,
        tree.Flag_EcalDeadCellTriggerPrimitiveFilter,
        tree.Flag_BadPFMuonFilter,
        tree.Flag_BadPFMuonDzFilter,
        tree.Flag_eeBadScFilter,
        tree.Flag_ecalBadCalibFilter
    ]):
        continue


    # trigger cuts
    passes_mu_trigger = any(
        getattr(tree, trig, False) for trig in cuts["Trigger"]["muon"]
        )

    passes_ele_trigger = any(
        getattr(tree, trig, False) for trig in cuts["Trigger"]["electron"]
        )   

    if passes_mu_trigger:
        cutflow["pass_mu_trigger"] += 1

    if passes_ele_trigger:
        cutflow["pass_ele_trigger"] += 1

    # muons

    muons = []
    for mu_idx in range(tree.nMuon):
        mu = MyMuon(
            tree.Muon_pt[mu_idx],
            tree.Muon_eta[mu_idx],
            tree.Muon_phi[mu_idx],
            tree.Muon_mass[mu_idx],
            tree.Muon_miniPFRelIso_all[mu_idx],
            tree.Muon_charge[mu_idx]
        )

        mu.reco_index = mu_idx
        mu.gen_idx = int(tree.Muon_genPartIdx[mu_idx])
        mu.kind = "muon"

        muons.append(mu)

    iso_muons = [
    m for m in muons
    if m.Pt() > cuts["Muon"]["pt_min"]
    and abs(m.Eta()) < cuts["Muon"]["eta_max"]
    and m.isolation < cuts["Muon"]["iso_max"]
    ]



    iso_muons = sorted(iso_muons, key=lambda m: m.Pt(), reverse=True)


    # electrons
    electrons = []
    for ele_idx in range(tree.nElectron):
        ele = MyElectron(
            tree.Electron_pt[ele_idx],
            tree.Electron_eta[ele_idx],
            tree.Electron_phi[ele_idx],
            tree.Electron_mass[ele_idx],
            tree.Electron_miniPFRelIso_all[ele_idx],
            tree.Electron_charge[ele_idx],
            tree.Electron_cutBased[ele_idx]
        )

        ele.reco_index = ele_idx
        ele.gen_idx = int(tree.Electron_genPartIdx[ele_idx])
        ele.kind = "electron"

        electrons.append(ele)
    
    iso_electrons = [
        e for e in electrons
        if e.Pt() > cuts["Electron"]["pt_min"]
        and abs(e.Eta()) < cuts["Electron"]["eta_max"]
        and e.cutBased >= cuts["Electron"]["cutBased"]
        and e.isolation < cuts["Electron"]["iso_max"] 
        ]

    iso_electrons = sorted(iso_electrons, key=lambda e: e.Pt(), reverse=True)

    # MET from tree
    MET  = tree.MET_pt
    phi  = tree.MET_phi


    METx = MET * r.TMath.Cos(phi)
    METy = MET * r.TMath.Sin(phi)

    n_iso_mu  = len(iso_muons)
    n_iso_ele = len(iso_electrons)

    if n_iso_mu == 1 and n_iso_ele == 0:
        if not passes_mu_trigger:
            continue
        selected_lepton = iso_muons[0]

    elif n_iso_ele == 1 and n_iso_mu == 0:
        if not passes_ele_trigger:
            continue
        selected_lepton = iso_electrons[0]

    else:
        continue

    cutflow["exactly_1_lepton"] += 1

    if selected_lepton.gen_idx < 0:
        reco_fail["selected_lepton_has_no_gen_match"] += 1
        continue

    if selected_lepton.gen_idx >= tree.nGenPart:
        reco_fail["selected_lepton_bad_gen_index"] += 1
        continue

    selected_gen_pdg = gen_pdg_ids[selected_lepton.gen_idx]

    if selected_lepton.kind == "muon":
        if abs(selected_gen_pdg) != 13:
            reco_fail["reco_muon_not_matched_to_gen_muon"] += 1
            continue

    elif selected_lepton.kind == "electron":
        if abs(selected_gen_pdg) != 11:
            reco_fail["reco_electron_not_matched_to_gen_electron"] += 1
            continue

    expected_charge = -1 if selected_gen_pdg > 0 else 1

    if int(selected_lepton.charge) != expected_charge:
        reco_fail["selected_lepton_charge_mismatch"] += 1
        continue

    if gen_W_lep_idx is None:
        reco_fail["missing_gen_leptonic_W"] += 1
        continue

    if not has_ancestor(
        selected_lepton.gen_idx,
        gen_W_lep_idx,
        gen_mothers
    ):
        reco_fail["selected_lepton_not_from_leptonic_W"] += 1
        continue

    passes_MET = MET > cuts["MET"]["pt_min"]

    if not passes_MET:
        continue
    cutflow["pass_MET"] += 1

    
    # jets

    jets = []
    for jet_idx in range(tree.nJet):
        jet = MyJet(
            tree.Jet_pt[jet_idx],
            tree.Jet_eta[jet_idx],
            tree.Jet_phi[jet_idx],
            tree.Jet_mass[jet_idx],
            tree.Jet_btagDeepFlavB[jet_idx],
            tree.Jet_jetId[jet_idx]
        )

        jet.reco_index = jet_idx

        jets.append(jet)

    good_jets = [
    j for j in jets
    if j.Pt() > cuts["Jets"]["pt_min"]
    and abs(j.Eta()) < cuts["Jets"]["eta_max"]
    and j.HasJetID()
    and j.DeltaR(selected_lepton) > 0.4
    ]

    #good_jets = [j for j in jets if j.HasJetID() and j.Pt() > JetPtCut]
    good_jets = sorted(good_jets, key=lambda j: j.Pt(), reverse=True)
    
    

    # flagging b-tagged jets
    for j in good_jets:
        j.is_btagged = j.IsBTagged(cuts["Jets"]["btag"])

    bjets = [j for j in good_jets if j.is_btagged]
    bjets = sorted(bjets, key=lambda j: j.Pt(), reverse=True)

    n_jets = len(good_jets)
    n_bjets = len(bjets)

    sigma2 = np.array([
        [100, 0],
        [0, 100]
    ])
    

    # 4-6 jets
    if 4 <= n_jets <= 6:
        cutflow["4plus_jets"] += 1

        if n_bjets < 2:
            continue

        cutflow["4pj_2b"] += 1

        four_jet_match = unique_jet_parton_match(
            good_jets,
            [
                gen_b_lep,
                gen_b_had,
                gen_w_quarks[0],
                gen_w_quarks[1]
            ],
            max_dr=0.4
        )

        if four_jet_match is None:
            reco_fail["four_partons_not_uniquely_matched"] += 1
            continue

        matched_b_lep = four_jet_match["jets"][0]
        matched_b_had = four_jet_match["jets"][1]
        matched_q1 = four_jet_match["jets"][2]
        matched_q2 = four_jet_match["jets"][3]

        if not matched_b_lep.is_btagged:
            reco_fail["matched_b_lep_not_btagged"] += 1
            continue

        if not matched_b_had.is_btagged:
            reco_fail["matched_b_had_not_btagged"] += 1
            continue

        W_had = matched_q1 + matched_q2
        t_had = W_had + matched_b_had

        if not np.isfinite(W_had.M()):
            reco_fail["nonfinite_fourjet_W_mass"] += 1
            continue

        if not np.isfinite(t_had.M()):
            reco_fail["nonfinite_fourjet_top_mass"] += 1
            continue

        h_Mwh_vs_Mth.Fill(
            t_had.M(),
            W_had.M(),
            1.0
        )

    # 3 jets
    if n_jets == 3:
        cutflow["3jets"] += 1

        # This template currently studies the 3-jet, 2-b-tag category.
        if n_bjets != 2:
            continue

        cutflow["3j_2b"] += 1

        b1, b2 = bjets[0], bjets[1]

        light_jets = [
            jet for jet in good_jets
            if not jet.is_btagged
        ]

        if len(light_jets) != 1:
            reco_fail["threejet_not_exactly_one_light_jet"] += 1
            continue

        light_jet = light_jets[0]

        # Check that b1 and b2 can be matched uniquely to b_lep and b_had.
        first_b_order_correct = (
            b1.DeltaR(gen_b_lep) < 0.4
            and
            b2.DeltaR(gen_b_had) < 0.4
        )

        second_b_order_correct = (
            b2.DeltaR(gen_b_lep) < 0.4
            and
            b1.DeltaR(gen_b_had) < 0.4
        )

        if first_b_order_correct == second_b_order_correct:
            reco_fail["threejet_b_matching_not_unique"] += 1
            continue

        light_matches_q1 = (
            light_jet.DeltaR(gen_w_quarks[0]) < 0.4
        )

        light_matches_q2 = (
            light_jet.DeltaR(gen_w_quarks[1]) < 0.4
        )

        # The light jet must match exactly one hadronic-W quark.
        if light_matches_q1 == light_matches_q2:
            reco_fail["threejet_W_jet_matching_not_unique"] += 1
            continue

        light_is_correct = True

        n_3jet_before_solver += 1

        assignments = []

        for b_lep, b_had in [
            (b1, b2),
            (b2, b1)
        ]:

            try:
                nusol = singleNeutrinoSolution(
                    b_lep,
                    selected_lepton,
                    METx,
                    METy,
                    sigma2
                )

            except Exception as error:
                n_3jet_solver_fail += 1

                if n_3jet_solver_fail <= 5:
                    print(
                        f"Solver failed in event {event}:",
                        error
                    )

                continue

            nu_components = np.asarray(
                nusol.nu,
                dtype=float
            ).reshape(-1)

            if len(nu_components) < 3:
                n_3jet_solver_fail += 1
                reco_fail["solver_returned_bad_neutrino"] += 1
                continue

            Dn = math.hypot(
                nu_components[0] - METx,
                nu_components[1] - METy
            )

            Mbhj = (b_had + light_jet).M()

            if not np.isfinite(Dn):
                reco_fail["nonfinite_Dn"] += 1
                continue

            if not np.isfinite(Mbhj):
                reco_fail["nonfinite_Mbhj"] += 1
                continue

            nu = make_neutrino(nusol)

            b_lep_correct = (
                b_lep.DeltaR(gen_b_lep) < 0.4
            )

            b_had_correct = (
                b_had.DeltaR(gen_b_had) < 0.4
            )

            assignment_correct = (
                b_lep_correct
                and b_had_correct
                and light_is_correct
            )

            # Dn distribution for the proposed leptonic-b candidate.
            if b_lep_correct:
                h_Dn_correct.Fill(Dn, 1.0)
            else:
                h_Dn_wrong.Fill(Dn, 1.0)

            # Mass distribution for proposed b_had + W jet.
            if b_had_correct and light_is_correct:
                h_Mbhj_correct.Fill(Mbhj, 1.0)
            else:
                h_Mbhj_wrong.Fill(Mbhj, 1.0)

            assignments.append({
                "b_lep": b_lep,
                "b_had": b_had,
                "light_jet": light_jet,
                "nu": nu,
                "Dn": Dn,
                "Mbhj": Mbhj,
                "truth_correct": assignment_correct
            })

            n_3jet_solver_success += 1

        if len(assignments) == 0:
            reco_fail["threejet_no_solver_assignment"] += 1
            continue

        correct_assignments = [
            assignment
            for assignment in assignments
            if assignment["truth_correct"]
        ]

        if len(correct_assignments) != 1:
            reco_fail["not_exactly_one_correct_assignment"] += 1
            continue



def normalize_hist(hist):
    integral = hist.Integral()

    if integral > 0:
        hist.Scale(1.0 / integral)

print("Mass correct:", h_Mbhj_correct.GetEntries())
print("Mass wrong:", h_Mbhj_wrong.GetEntries())
print("3-jet events before solver:", n_3jet_before_solver)
print("3-jet solver successes:", n_3jet_solver_success)
print("3-jet solver failures:", n_3jet_solver_fail)

print("\n=== GENERATOR TRUTH FAILURES ===")

for failure_name, failure_count in truth_fail.items():
    print(f"{failure_name:40s}: {failure_count}")

print("\n=== RECONSTRUCTION FAILURES ===")

for failure_name, failure_count in reco_fail.items():
    print(f"{failure_name:45s}: {failure_count}")

for hist in [
    h_Mbhj_correct,
    h_Mbhj_wrong
]:
    normalize_hist(hist)

# histograms

print("\n=== CUTFLOW ===")
for key, val in cutflow.items():
    print(f"{key:15s}: {val}")



#probability 2d hist

c_Mwh_vs_Mth = r.TCanvas("c_Mwh_vs_Mth", "M(W_h) vs M(t_h)", 800, 600)

total = h_Mwh_vs_Mth.Integral()

if total > 0:
    h_Mwh_vs_Mth.Scale(1.0 / total)

print("Probability sum:", h_Mwh_vs_Mth.Integral())

def top_probability(M_top, M_W, hist):
    return max(0.0, hist.Interpolate(M_top, M_W))


#p = top_probability(173.2, 81.1, h_Mwh_vs_Mth)
#print(p)

h_Mwh_vs_Mth.Draw("COLZ")

legend = r.TLegend(0.15, 0.80, 0.45, 0.88)
legend.SetBorderSize(0)
legend.SetFillStyle(0)
legend.AddEntry(h_Mwh_vs_Mth,
                "Color = probability",
                "f")
c_Mwh_vs_Mth.Update()


import os
os.makedirs("plots", exist_ok=True)
c_Mwh_vs_Mth.SaveAs("plots/h_Mwh_vs_Mth.png")

canvases = []


def save_comparison(
    correct_hist,
    wrong_hist,
    canvas_name,
    output_name
):
    canvas = r.TCanvas(canvas_name, canvas_name, 800, 600)
    canvases.append(canvas)

    correct_hist.SetLineColor(r.kBlue)
    wrong_hist.SetLineColor(r.kRed)

    correct_hist.SetLineWidth(2)
    wrong_hist.SetLineWidth(2)

    if correct_hist.GetMaximum() >= wrong_hist.GetMaximum():
        correct_hist.Draw("HIST")
        wrong_hist.Draw("HIST SAME")
    else:
        wrong_hist.Draw("HIST")
        correct_hist.Draw("HIST SAME")

    legend = r.TLegend(0.58, 0.68, 0.88, 0.88)
    legend.SetBorderSize(0)
    legend.SetFillStyle(0)

    legend.AddEntry(correct_hist,
                    "Blue: correct assignment",
                    "l")
    legend.AddEntry(wrong_hist,
                    "Red: wrong assignment",
                    "l")

    legend.Draw()

    canvas.Update()
    canvas.Draw()
    canvas.SaveAs(output_name)

    return canvas


c_Dn_leptonic = save_comparison(
    h_Dn_correct,
    h_Dn_wrong,
    "c_Dn_leptonic",
    "plots/Dn_leptonic_b.png"
)

c_Mbhj = save_comparison(
    h_Mbhj_correct,
    h_Mbhj_wrong,
    "c_Mbhj",
    "plots/Mbhj_3jet.png"
)

Events in file: 197164

========== GENERATOR EVENT 0 ==========
idx=  0  pdg=   21  mother_idx= -1  mother_pdg= None  lastCopy=False
idx=  1  pdg=   21  mother_idx= -1  mother_pdg= None  lastCopy=False
idx=  2  pdg=   -6  mother_idx=  0  mother_pdg=   21  lastCopy=False
idx=  3  pdg=    6  mother_idx=  0  mother_pdg=   21  lastCopy=False
idx=  4  pdg=   21  mother_idx=  0  mother_pdg=   21  lastCopy=False
idx=  5  pdg=   -6  mother_idx=  2  mother_pdg=   -6  lastCopy=True
idx=  6  pdg=    6  mother_idx=  3  mother_pdg=    6  lastCopy=True
idx=  7  pdg=  -24  mother_idx=  5  mother_pdg=   -6  lastCopy=False
idx=  8  pdg=   -5  mother_idx=  5  mother_pdg=   -6  lastCopy=False
idx=  9  pdg=  -24  mother_idx=  7  mother_pdg=  -24  lastCopy=True
idx= 10  pdg=   24  mother_idx=  6  mother_pdg=    6  lastCopy=False
idx= 11  pdg=    5  mother_idx=  6  mother_pdg=    6  lastCopy=False
idx= 12  pdg=   24  mother_idx= 10  mother_pdg=   24  lastCopy=True
idx= 13  pdg=   15  mother_idx=  9  mother_

Solver failed in event 1781: Singular matrix A
Solver failed in event 1872: Singular matrix A
Mass correct: 4100.0
Mass wrong: 2632.0
3-jet events before solver: 4170
3-jet solver successes: 6732
3-jet solver failures: 1608

=== GENERATOR TRUTH FAILURES ===
top_does_not_have_exact_expected_W      : 3028

=== RECONSTRUCTION FAILURES ===
threejet_b_matching_not_unique               : 667
four_partons_not_uniquely_matched            : 6734
matched_b_lep_not_btagged                    : 208
selected_lepton_not_from_leptonic_W          : 96
matched_b_had_not_btagged                    : 219
selected_lepton_has_no_gen_match             : 125
threejet_W_jet_matching_not_unique           : 678
selected_lepton_charge_mismatch              : 58
threejet_no_solver_assignment                : 32
reco_electron_not_matched_to_gen_electron    : 10
not_exactly_one_correct_assignment           : 39

=== CUTFLOW ===
total          : 197164
pass_mu_trigger: 38360
pass_ele_trigger: 32643
exactly_1_lepton:

Warning in <TClass::Init>: no dictionary for class edm::Hash<1> is available
Warning in <TClass::Init>: no dictionary for class edm::ProcessHistory is available
Warning in <TClass::Init>: no dictionary for class edm::ProcessConfiguration is available
Warning in <TClass::Init>: no dictionary for class edm::ParameterSetBlob is available
Warning in <TClass::Init>: no dictionary for class pair<edm::Hash<1>,edm::ParameterSetBlob> is available
Info in <TCanvas::Print>: png file plots/h_Mwh_vs_Mth.png has been created
Info in <TCanvas::Print>: png file plots/Dn_leptonic_b.png has been created
Info in <TCanvas::Print>: png file plots/Mbhj_3jet.png has been created
